# 10. MultiIndex & Hierarchical Indexing: Beginner Guide

### 🌟 What is Hierarchical Multi-Level Indexing (`MultiIndex`) in Pandas?
A **MultiIndex** allows you to store higher-dimensional data inside a standard 2D DataFrame. This notebook covers constructing multi-level row and column indices, slicing across hierarchical levels, and extracting cross-sections with `.xs()`.

This interactive guide loads and works directly with `data/raw_transactions.csv`, giving you real-world hands-on practice.

### 📚 Key Concepts Covered in this Notebook:
- **Construction & Swapping**: Covers `pd.MultiIndex.from_tuples()` and `.swaplevel()`.
- **Multi-Level Slicing**: Covers `.loc[pd.IndexSlice]`.
- **Level Aggregations**: Covers aggregating across index levels.


In [1]:
# Setup imports & dataset loading
import pandas as pd
import numpy as np
import sys
import time
import os
import matplotlib.pyplot as plt

# Load raw transactions dataset
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
df = pd.read_csv(csv_path)
print(f"Pandas Version: {pd.__version__}")
print(f"Loaded raw_transactions.csv: {df.shape[0]} rows, {df.shape[1]} columns")

Pandas Version: 2.2.2
Loaded raw_transactions.csv: 15000 rows, 11 columns


### 🔹 MultiIndex Construction from Groupby
Constructs a hierarchical MultiIndex DataFrame grouping by region and card_type. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** After running `.groupby().agg()`, the grouped columns become index levels. Call `.reset_index()` to bring them back as standard columns.

**Syntax:** `df.groupby(['region', 'card_type'])[['transaction_amount', 'is_fraud']].mean()`


In [2]:
multi_df = df.groupby(['region', 'card_type'])[['transaction_amount', 'is_fraud']].mean()
print('MultiIndexed Financial Summary:\n', multi_df.round(3))

MultiIndexed Financial Summary:
                     transaction_amount  is_fraud
region  card_type                               
 East   Amex                  1003.788     0.000
        Discover               810.773     0.050
        MasterCard            1016.641     0.059
        Visa                   905.949     0.105
 North  Amex                  1167.633     0.167
        Discover              1234.215     0.174
        MasterCard            1126.083     0.056
        Visa                   894.139     0.000
 South  Amex                   965.991     0.129
        Discover              1052.486     0.176
        MasterCard            1192.082     0.048
        Visa                  1006.453     0.062
 West   Amex                   925.648     0.083
        Discover               976.084     0.143
        MasterCard            1037.675     0.059
        Visa                   742.779     0.045
East    Amex                  1011.577     0.103
        Discover              1007.4

### 🔹 Swapping MultiIndex Levels with `.swaplevel()`
Swaps region and card_type index levels. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `multi_df.swaplevel(0, 1)`


In [3]:
swapped_levels = multi_df.swaplevel('region', 'card_type')
print('Swapped Levels:\n', swapped_levels.head(4))

Swapped Levels:
                    transaction_amount  is_fraud
card_type  region                              
Amex       East           1003.787647  0.000000
Discover   East            810.773158  0.050000
MasterCard East           1016.640588  0.058824
Visa       East            905.949444  0.105263


### 🔹 Cross-Level Slicing with `pd.IndexSlice`
Extracts all metrics for 'Visa' cards across all global regions. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Remember: Basic slicing creates a *view* into the original array. Modifying a view changes the original array! Use `.copy()` when you need an isolated duplicate.

**Syntax:** `multi_df.loc[pd.IndexSlice[:, 'Visa'], :]`


In [4]:
idx = pd.IndexSlice
visa_slice = multi_df.loc[idx[:, 'Visa'], :]
print('Visa Slices Across All Regions:\n', visa_slice)

Visa Slices Across All Regions:
                    transaction_amount  is_fraud
region  card_type                              
 East   Visa               905.949444  0.105263
 North  Visa               894.138636  0.000000
 South  Visa              1006.453125  0.062500
 West   Visa               742.778571  0.045455
East    Visa               982.976262  0.094595
North   Visa              1007.937580  0.113537
South   Visa               999.274281  0.113895
West    Visa              1000.855976  0.101617
east    Visa              1014.073636  0.136364
north   Visa              1195.613500  0.200000
south   Visa              1130.910556  0.105263
west    Visa               953.855294  0.052632


### 🔹 Aggregating Across MultiIndex Levels
Computes top-level region totals collapsing card_type level. Pandas simplifies tabular workflows, offering expressive one-line operations for filtering, aggregating, joining, and cleaning datasets. **Tip:** Inspect your data types early with `df.dtypes` to ensure numeric values weren't accidentally parsed as text.

**Syntax:** `multi_df.groupby(level='region').mean()`


In [5]:
region_collapsed = multi_df.groupby(level='region').mean()
print('Collapsed Region Level Averages:\n', region_collapsed)

Collapsed Region Level Averages:
          transaction_amount  is_fraud
region                               
 East            934.287709  0.053522
 North          1105.517383  0.099034
 South          1054.253044  0.103905
 West            920.546260  0.082617
East             995.984612  0.098752
North           1017.627913  0.112686
South           1002.802558  0.104998
West             998.414059  0.105748
east            1148.740266  0.173077
north           1097.966669  0.173191
south           1147.452917  0.109279
west            1101.572829  0.096491


## 💡 Real-World Practice & Scenarios
Practical scenarios and common data questions explained simply with real examples.


### 🔍 Scenario: Q1: Cross-Sectional MultiIndex Lookup with `.xs()`

**Approach:** Extract all regional statistics for 'MasterCard' using `.xs()` without resetting index.
**Syntax:** `multi_df.xs('MasterCard', level='card_type')`


In [6]:
mc_xs = multi_df.xs('MasterCard', level='card_type')
print('MasterCard Cross-Section (.xs):\n', mc_xs)

MasterCard Cross-Section (.xs):
          transaction_amount  is_fraud
region                               
 East           1016.640588  0.058824
 North          1126.083333  0.055556
 South          1192.082500  0.047619
 West           1037.674706  0.058824
East             981.911304  0.095085
North           1021.453775  0.108597
South           1024.759238  0.120551
West             980.810181  0.105717
east            1336.467619  0.181818
north            931.937222  0.105263
south           1254.934500  0.136364
west             906.028571  0.000000
